# Visualización Interactiva de Curvas Elípticas (ECC)

Este notebook explora la matemática detrás de la criptografía de curva elíptica y cómo se derivan las identidades digitales en Ethereum a partir de frases mnemotécnicas.

### 1. Instalación de Dependencias

In [1]:
# Instalamos librerías para interactividad, criptografía y manejo de mnemonics
!pip install fastecdsa matplotlib numpy plotly ipywidgets mnemonic eth-account

### 2. Curva Elíptica Continua (Interactiva)

Usa los deslizadores para ver cómo cambian los parámetros $a$ y $b$ la forma de la curva:
$$y^2 = x^3 + ax + b$$

In [5]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider

def update_plot(a, b):
    y, x = np.ogrid[-10:10:200j, -10:10:200j]
    z = y**2 - (x**3 + a*x + b)
    
    fig = go.Figure(data=go.Contour(
        z=z.flatten(),
        x=np.linspace(-10, 10, 200),
        y=np.linspace(-10, 10, 200),
        contours_coloring='lines',
        line_width=3,
        contours=dict(start=0, end=0, size=1),
        line_color='teal'
    ))
    
    fig.update_layout(
        title=f'Curva Elíptica: y^2 = x^3 + ({a})x + ({b})',
        xaxis_title='Eje X',
        yaxis_title='Eje Y',
        width=700, height=600, template='plotly_dark'
    )
    fig.show()

interact(update_plot, 
         a=FloatSlider(value=0, min=-5, max=5, step=0.1, description='a'), 
         b=FloatSlider(value=7, min=-10, max=10, step=0.1, description='b'))

interactive(children=(FloatSlider(value=0.0, description='a', max=5.0, min=-5.0), FloatSlider(value=7.0, descr…

<function __main__.update_plot(a, b)>

### 3. Puntos en Cuerpo Finito (Visualización Dinámica)

En la práctica blockchain, los puntos son discretos. Selecciona un número primo $p$ para ver la "nube de puntos" resultante.

In [3]:
def plot_finite_interactive(a, b, p):
    x_points = []
    y_points = []
    for x in range(p):
        rhs = (x**3 + a*x + b) % p
        for y in range(p):
            if (y**2) % p == rhs:
                x_points.append(x)
                y_points.append(y)
    
    fig = go.Figure(data=go.Scatter(
        x=x_points, y=y_points, mode='markers', 
        marker=dict(size=10, color='crimson', line=dict(width=1, color='DarkSlateGrey'))
    ))
    
    fig.update_layout(
        title=f'Puntos de la Curva sobre el Campo Finito F_{p}',
        xaxis_title='X', yaxis_title='Y', width=800, height=600, template='plotly_dark'
    )
    fig.show()

interact(plot_finite_interactive, 
         a=FloatSlider(value=0, min=-2, max=2, step=1, description='a'), 
         b=FloatSlider(value=7, min=0, max=10, step=1, description='b'),
         p=[17, 31, 61, 97, 127, 251])

interactive(children=(FloatSlider(value=0.0, description='a', max=2.0, min=-2.0, step=1.0), FloatSlider(value=…

<function __main__.plot_finite_interactive(a, b, p)>

### 4. Generación de Wallets: Frases Mnemotécnicas (BIP-39)

En lugar de recordar claves privadas aleatorias, utilizamos frases de 12 o 24 palabras. Estas palabras representan la entropía necesaria para reconstruir todas las claves de una billetera.

In [4]:
from mnemonic import Mnemonic
from eth_account import Account

# 1. Generar Frase Mnemotécnica (12 palabras en español)
mnemo = Mnemonic("spanish")
words = mnemo.generate(strength=128)  # 128 bits = 12 palabras
print(f"📦 Frase Semilla (Mnemonic):\n{words}\n")

# 2. Habilitar funciones HD (Jerárquicas Determínisticas)
Account.enable_unaudited_hdwallet_features()

# 3. Derivar Clave Privada usando la ruta estándar BIP-44 para Ethereum
# Ruta: m / purpose' / coin_type' / account' / change / address_index
path = "m/44'/60'/0'/0/0"
account = Account.from_mnemonic(words, account_path=path)

print(f"🔑 Clave Privada Derivada (hex):\n{account.key.hex()}\n")
print(f"💳 Dirección Ethereum (0x):\n{account.address}")

📦 Frase Semilla (Mnemonic):
linterna núcleo preso ajuste sorpresa muro cueva dragón vago broma unir alerta

🔑 Clave Privada Derivada (hex):
170f43a275c942204e692d4077493503c4685db038ce3322baa4813c59c386bb

💳 Dirección Ethereum (0x):
0x1D04c1FC2d17738240406E96884E4F7dc49e71e4


### 5. ¿Cómo pasamos de la frase a la dirección?

1. **Entropía:** Generamos 128 bits aleatorios.
2. **Palabras (BIP-39):** Convertimos esos bits en 12 palabras de una lista estandarizada.
3. **Semilla (Seed):** Usamos PBKDF2 para convertir las palabras en una semilla de 512 bits.
4. **Derivación (BIP-32/44):** Usamos HMAC-SHA512 para derivar claves hijo a partir de la semilla.
5. **ECC (secp256k1):** La clave privada derivada se multiplica por el punto generador $G$ para obtener la clave pública.
6. **Dirección (Keccak-256):** El hash de la clave pública nos da los últimos 20 bytes, que son la dirección de Ethereum.